# 02 Procesamiento de texto, recuperación secuencial e indexación

## Objetivo

Implementar y comparar dos mecanismos de recuperación de información —búsqueda secuencial e índice invertido— sobre colecciones de libros del Proyecto Gutenberg, investigando experimentalmente su rendimiento y escalabilidad.

### Parte A. Corpus pequeño

**1. Preparación del corpus**

Utilice los 18 documentos de la colección Gutenberg disponible en NLTK.

1. Cargue los textos originales.
2. Identifique el número de documentos y sus tamaños.
3. Implemente tokenización y normalización: conversión a minúsculas, tratamiento de puntuación y caracteres especiales.

Seleccione y documente una configuración de preprocesamiento que se utilizará consistentemente en todos los experimentos.

**Carga del dataset e identificación de documentos**

In [1]:
from pathlib import Path
import nltk
from nltk.corpus import gutenberg

root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
ruta = str(root_dir / "data" / "nltk_data")
if ruta not in nltk.data.path:
    nltk.data.path.insert(0, ruta)

documentos = gutenberg.fileids()
print("Cantidad:", len(documentos))

for doc_id in documentos:
    kb = Path(gutenberg.abspath(doc_id)).stat().st_size / 1024
    print(f"{doc_id}: {kb:.1f} KB")

Cantidad: 18
austen-emma.txt: 866.3 KB
austen-persuasion.txt: 455.4 KB
austen-sense.txt: 657.2 KB
bible-kjv.txt: 4231.0 KB
blake-poems.txt: 37.3 KB
bryant-stories.txt: 243.6 KB
burgess-busterbrown.txt: 82.7 KB
carroll-alice.txt: 141.0 KB
chesterton-ball.txt: 446.7 KB
chesterton-brown.txt: 397.1 KB
chesterton-thursday.txt: 313.0 KB
edgeworth-parents.txt: 913.2 KB
melville-moby_dick.txt: 1213.9 KB
milton-paradise.txt: 457.2 KB
shakespeare-caesar.txt: 109.7 KB
shakespeare-hamlet.txt: 159.1 KB
shakespeare-macbeth.txt: 98.0 KB
whitman-leaves.txt: 694.5 KB


**Tokenización y limpieza de caracteres**

Configuración aplicada en todos los experimentos:
- Se convierten los textos a minúsculas antes de tokenizar.
- `RegexpTokenizer(r"[A-Za-z]+(?:['’][A-Za-z]+)?")` conserva secuencias de letras ASCII y apóstrofos internos; otros signos, incluidos los guiones, separan tokens.
- No se eliminan stopwords ni se aplica stemming o lematización.
- El patrón no incluye dígitos ni letras acentuadas; esta limitación se mantiene igual para los documentos y las consultas.

- Se reutiliza la misma tokenización para cada operando de consulta; se rechazan entradas sin tokens o con más de uno. AND y OR reciben dos operandos y NOT recibe uno.


In [59]:
from nltk.tokenize import RegexpTokenizer

tokenizador = RegexpTokenizer(r"[A-Za-z]+(?:['’][A-Za-z]+)?")


def tokenizar(texto):
    return tokenizador.tokenize(texto.lower())


def normalizar_termino(termino):
    termino = termino.strip().lower()
    # Un término formado solo por letras ASCII ya es un token válido.
    if termino.isascii() and termino.isalpha():
        return termino
    tokens = tokenizar(termino)
    if len(tokens) != 1:
        raise ValueError("Cada operando debe contener un único término válido.")
    return tokens[0]


corpus_procesado = {}
for doc_id in documentos:
    corpus_procesado[doc_id] = tokenizar(gutenberg.raw(doc_id))

**2. Búsqueda secuencial**

Implemente una función que reciba un término de consulta y devuelva los identificadores de todos los documentos que lo contienen con una búsqueda secuencial.

La implementación deberá utilizar obligatoriamente dos lazos `for` anidados: el primero recorre los documentos y el segundo los tokens de cada documento.

No es necesario utilizar índices invertidos, motores de búsqueda ni estructuras auxiliares que almacenen anticipadamente los documentos asociados a cada término.

Ejecute consultas con términos frecuentes, poco frecuentes e inexistentes.

In [60]:
def busqueda_secuencial(termino):
    termino = normalizar_termino(termino)
    resultado = []

    for doc_id, tokens in corpus_procesado.items():
        for token in tokens:
            if token == termino:
                resultado.append(doc_id)
                break
    return resultado

In [61]:
consultas = {
    "frecuente": "he",
    "poco frecuente": "moby",
    "inexistente": "tinitana",
}

for categoria, termino in consultas.items():
    resultados = busqueda_secuencial(termino)
    print(f"{categoria} ({termino}): ",resultados)

frecuente (he):  ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']
poco frecuente (moby):  ['melville-moby_dick.txt']
inexistente (tinitana):  []


**3. Medición de tiempos de ejecución**

Utilice `time.perf_counter_ns()` o una herramienta de medición equivalente para obtener el tiempo de ejecución de las consultas.

Cada medición deberá corresponder a **30 ejecuciones de la misma consulta**. Y se debe calcular la media aritmética.

Registre los resultados en milisegundos. No incluya la carga de archivos ni la tokenización dentro del tiempo de consulta.

In [62]:
import time


def medir_consulta(funcion_busqueda, *terminos, repeticiones=30):
    if repeticiones <= 0:
        raise ValueError("El número de repeticiones debe ser positivo.")

    tiempos_ms = []
    for _ in range(repeticiones):
        inicio = time.perf_counter_ns()
        funcion_busqueda(*terminos)
        fin = time.perf_counter_ns()
        tiempos_ms.append((fin - inicio) / 1_000_000)

    return sum(tiempos_ms) / repeticiones

In [63]:
for categoria, termino in consultas.items():
    promedio_ms = medir_consulta(busqueda_secuencial, termino, repeticiones=30)
    cantidad_documentos = len(busqueda_secuencial(termino))
    print(
        f"{categoria} ({termino}): {cantidad_documentos} documentos; "
        f"promedio = {promedio_ms:.4f} ms"
    )

frecuente (he): 18 documentos; promedio = 0.1413 ms
poco frecuente (moby): 1 documentos; promedio = 38.6279 ms
inexistente (tinitana): 0 documentos; promedio = 39.0692 ms


**4. Construcción del índice invertido**

Implemente un índice invertido que contenga el vocabulario de la colección y las listas de documentos asociados a cada término.

Inspeccione las listas de postings y calcule la frecuencia documental (DF) de varios términos.

Implemente consultas de un término y consultas booleanas con AND, OR y NOT.


Los postings se almacenan como conjuntos para aplicar directamente intersección (`&`, AND), unión (`|`, OR) y diferencia respecto al universo (`-`, NOT). El universo se calcula una vez. Las consultas devuelven listas ordenadas, sin modificar el índice; el ordenamiento de la salida también forma parte del tiempo medido.

Los conjuntos evitan reconstruir los postings en cada consulta, a cambio de mayor memoria que las listas ordenadas. Para el corpus grande habrá que medir también este costo.


In [64]:
from collections import defaultdict

# Los conjuntos se construyen una sola vez; cada documento aparece una vez por término.
indice_invertido = defaultdict(set)
universo_documentos = set(corpus_procesado)
postings_vacios = frozenset()

for doc_id, tokens in corpus_procesado.items():
    for termino in set(tokens):
        indice_invertido[termino].add(doc_id)

In [65]:
import pandas as pd

# La matriz es solo una visualización: materializamos únicamente la muestra.
terminos_muestra = sorted(indice_invertido)[:18]
documentos_muestra = list(corpus_procesado)[:18]
matriz = pd.DataFrame(
    [
        [doc_id in indice_invertido[termino] for doc_id in documentos_muestra]
        for termino in terminos_muestra
    ],
    index=terminos_muestra,
    columns=documentos_muestra,
    dtype="int8",
)

print(
    "Tamaño completo (términos, documentos):",
    (len(indice_invertido), len(universo_documentos)),
)
print("Muestra materializada:", matriz.shape)
display(matriz)

Tamaño completo (términos, documentos): (43495, 18)
Muestra materializada: (18, 18)


,austen-emma.txt,austen-persuasion.txt,austen-sense.txt,bible-kjv.txt,blake-poems.txt,bryant-stories.txt,burgess-busterbrown.txt,carroll-alice.txt,chesterton-ball.txt,chesterton-brown.txt,chesterton-thursday.txt,edgeworth-parents.txt,melville-moby_dick.txt,milton-paradise.txt,shakespeare-caesar.txt,shakespeare-hamlet.txt,shakespeare-macbeth.txt,whitman-leaves.txt
a,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
a'lee,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
a'low,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
a'm,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
a'most,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0
a'n,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
a'ready,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
a'shiver,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
a'th,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
a'top,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0


In [66]:
def consulta_termino(termino):
    return sorted(indice_invertido.get(normalizar_termino(termino), postings_vacios))


def consulta_and(termino_1, termino_2):
    postings_1 = indice_invertido.get(normalizar_termino(termino_1), postings_vacios)
    postings_2 = indice_invertido.get(normalizar_termino(termino_2), postings_vacios)
    return sorted(postings_1 & postings_2)


def consulta_or(termino_1, termino_2):
    postings_1 = indice_invertido.get(normalizar_termino(termino_1), postings_vacios)
    postings_2 = indice_invertido.get(normalizar_termino(termino_2), postings_vacios)
    return sorted(postings_1 | postings_2)


def consulta_not(termino):
    postings = indice_invertido.get(normalizar_termino(termino), postings_vacios)
    return sorted(universo_documentos - postings)


terminos_df = ["he", "science", "moby", "tinitana"]
resumen_postings = []
for termino in terminos_df:
    postings = consulta_termino(termino)
    resumen_postings.append({"término": termino, "postings": postings, "DF": len(postings)})
display(pd.DataFrame(resumen_postings))

print("Consulta de un término (moby):", consulta_termino("moby"))
print("science AND whale:", consulta_and("science", "whale"))
print("science OR whale:", consulta_or("science", "whale"))
print("NOT science:", consulta_not("science"))

,término,postings,DF
0,he,"[austen-emma.txt, austen-persuasion.txt, auste...",18
1,science,"[bible-kjv.txt, chesterton-ball.txt, chesterto...",8
2,moby,[melville-moby_dick.txt],1
3,tinitana,[],0


Consulta de un término (moby): ['melville-moby_dick.txt']
science AND whale: ['bible-kjv.txt', 'chesterton-ball.txt', 'melville-moby_dick.txt', 'whitman-leaves.txt']
science OR whale: ['bible-kjv.txt', 'bryant-stories.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-hamlet.txt', 'whitman-leaves.txt']
NOT science: ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt']


**5. Comparación**

Ejecute las mismas consultas con ambos métodos.

Compruebe que recuperan exactamente los mismos identificadores de documentos y mida el tiempo medio de 30 ejecuciones para cada consulta y método.

La carga del corpus, la tokenización y la construcción del índice ocurren antes de medir las consultas.


Se comparan consultas simples y booleanas, incluidos términos inexistentes. Para las booleanas secuenciales se combinan los resultados de las búsquedas del punto 2, sin consultar el índice. La igualdad se comprueba por identificadores, independientemente de su orden, y una discrepancia detiene la comparación. Los tiempos son medias por consulta; no incluyen la presentación de la tabla.


In [ ]:
def consulta_and_secuencial(termino_1, termino_2):
    documentos_1 = set(busqueda_secuencial(termino_1))
    documentos_2 = set(busqueda_secuencial(termino_2))
    return sorted(documentos_1 & documentos_2)


def consulta_or_secuencial(termino_1, termino_2):
    documentos_1 = set(busqueda_secuencial(termino_1))
    documentos_2 = set(busqueda_secuencial(termino_2))
    return sorted(documentos_1 | documentos_2)


def consulta_not_secuencial(termino):
    return sorted(set(corpus_procesado) - set(busqueda_secuencial(termino)))


casos_comparacion = [
    (f"{categoria} ({termino})", busqueda_secuencial, consulta_termino, (termino,))
    for categoria, termino in consultas.items()
]
casos_comparacion.extend([
    ("science AND whale", consulta_and_secuencial, consulta_and, ("science", "whale")),
    ("science OR whale", consulta_or_secuencial, consulta_or, ("science", "whale")),
    ("NOT science", consulta_not_secuencial, consulta_not, ("science",)),
    ("science AND tinitana", consulta_and_secuencial, consulta_and, ("science", "tinitana")),
    ("science OR tinitana", consulta_or_secuencial, consulta_or, ("science", "tinitana")),
    ("NOT tinitana", consulta_not_secuencial, consulta_not, ("tinitana",)),
])

comparacion = []
for consulta, metodo_secuencial, metodo_indice, terminos in casos_comparacion:
    resultados_secuencial = metodo_secuencial(*terminos)
    resultados_indice = metodo_indice(*terminos)
    mismos_documentos = set(resultados_secuencial) == set(resultados_indice)
    assert mismos_documentos, f"Los métodos difieren para: {consulta}"

    comparacion.append({
        "Consulta": consulta,
        "Mismos documentos": mismos_documentos,
        "Documentos": len(resultados_indice),
        "Secuencial (ms)": medir_consulta(metodo_secuencial, *terminos, repeticiones=30),
        "Índice (ms)": medir_consulta(metodo_indice, *terminos, repeticiones=30),
    })

comparacion = pd.DataFrame(comparacion)
display(comparacion.round(6))

,Consulta,Mismos documentos,Documentos,Secuencial (ms),Índice (ms)
0,frecuente (he),True,18,0.158220,0.001230
1,poco frecuente (moby),True,1,38.622477,0.000820
2,inexistente (tinitana),True,0,40.127133,0.000777
3,science AND whale,True,4,46.539577,0.001237
4,science OR whale,True,10,47.129707,0.001427
5,NOT science,True,10,22.647733,0.001280
6,science AND tinitana,True,0,61.680313,0.001113
7,science OR tinitana,True,8,63.144373,0.001457
8,NOT tinitana,True,18,39.292800,0.001500


### Parte B. Corpus grande

**6. Descarga de Gutenberg**

Descargue el archivo oficial `txt-files.tar.zip`, de aproximadamente 11 GB, desde:

https://www.gutenberg.org/cache/epub/feeds/

Identifique los archivos TXT, evite contar distintas versiones del mismo libro como documentos independientes y registre la cantidad de documentos válidos.

**7. Recuperación sobre el corpus grande**

Reutilice las funciones desarrolladas en la Parte A.

1. Procese los documentos con la misma configuración de normalización.
2. Ejecute las consultas mediante búsqueda secuencial y mida los tiempos.
3. Construya el índice invertido.
4. Repita las consultas utilizando el índice.
5. Compruebe la igualdad de resultados y compare los tiempos.

Trabaje con subconjuntos de la colección de tamaños crecientes, conservando los identificadores originales. Documente los recursos de memoria y almacenamiento utilizados.

In [3]:
import zipfile
import tarfile

carpeta = root_dir / "data" / "large_gutenberg"
archivo_zip = carpeta / "txt-files.tar.zip"
archivo_tar = carpeta / "txt-files.tar"
carpeta_textos = carpeta / "txt"

print("Extrayendo el ZIP...")
with zipfile.ZipFile(archivo_zip) as comprimido:
    comprimido.extract("txt-files.tar", path=carpeta)

print("Extrayendo los archivos del TAR...")
with tarfile.open(archivo_tar, "r:") as comprimido:
    comprimido.extractall(path=carpeta_textos, filter="data")

print("Extracción terminada:", carpeta_textos)

Extrayendo el ZIP...
Extrayendo los archivos del TAR...
Extracción terminada: C:\Users\johan\Desktop\UNI\7 Semestre\Recuperación_de_Información\data\large_gutenberg\txt


### Parte C. Análisis experimental de escalabilidad

**8. Tiempo de ejecución en función del corpus**

Seleccione al menos cinco tamaños crecientes del corpus grande. Como referencia, pueden utilizarse 50, 100, 250, 500 y 1.000 documentos, ampliándolos si el equipo lo permite.

Para cada tamaño $n$:

1. Ejecute una misma consulta mediante búsqueda secuencial.
2. Repítala 30 veces y calcule el tiempo promedio.
3. Construya el índice invertido correspondiente a ese subconjunto.
4. Ejecute la consulta mediante el índice, también 30 veces.
5. Registre ambos tiempos y el número total de tokens.

Elabore una **gráfica del tiempo medio de ejecución en función del número de documentos**, colocando el número de documentos en el eje X y el tiempo en milisegundos en el eje Y. Represente los dos métodos en la misma gráfica.

**9. Estimación de una función de tiempo**

Utilizando los resultados experimentales, determine una función:

$t=f(n)$

para cada método de recuperación.

Como primera aproximación, utilice un ajuste lineal por mínimos cuadrados:

$t(n)=an+b$

Estime los parámetros $a$ y $b$, reporte el coeficiente de determinación $R^2$ y represente las funciones ajustadas junto a los datos experimentales.

Explique si el modelo lineal representa adecuadamente las mediciones y cómo se relaciona con la complejidad computacional de cada algoritmo.